In [0]:
%run "/Workspace/ETL_ARQUITETURA_MEDALHAO/00.config/config"

Catálogo: workspace
Schemas: bronze_economia silver_economia gold_economia


In [0]:

import requests, pandas as pd
from pyspark.sql.functions import current_timestamp

url = "https://api.bcb.gov.br/dados/serie/bcdata.sgs.433/dados"
params = {"formato": "json", "dataInicial": "01/01/2024", "dataFinal": "31/12/2025"}

df_pd = pd.DataFrame(requests.get(url, params=params).json())
df_pd.columns = ["data", "ipca"]
df_pd["ipca"] = df_pd["ipca"].str.replace(",", ".").astype(float)

In [0]:

df_pd.display()

data,ipca
01/01/2024,0.42
01/02/2024,0.83
01/03/2024,0.16
01/04/2024,0.38
01/05/2024,0.46
01/06/2024,0.21
01/07/2024,0.38
01/08/2024,-0.02
01/09/2024,0.44
01/10/2024,0.56


In [0]:
df_pd = spark.createDataFrame(df_pd).withColumn("data_coleta", current_timestamp())

In [0]:
df_pd.display()

data,ipca,data_coleta
01/01/2024,0.42,2026-05-07T01:58:42.827Z
01/02/2024,0.83,2026-05-07T01:58:42.827Z
01/03/2024,0.16,2026-05-07T01:58:42.827Z
01/04/2024,0.38,2026-05-07T01:58:42.827Z
01/05/2024,0.46,2026-05-07T01:58:42.827Z
01/06/2024,0.21,2026-05-07T01:58:42.827Z
01/07/2024,0.38,2026-05-07T01:58:42.827Z
01/08/2024,-0.02,2026-05-07T01:58:42.827Z
01/09/2024,0.44,2026-05-07T01:58:42.827Z
01/10/2024,0.56,2026-05-07T01:58:42.827Z


In [0]:
df_pd.write.format("delta").mode("overwrite").saveAsTable(f"{CATALOG}.{BRONZE}.ipca")